# 7-box Sokoban: original repository protocol
Enable GPU, attach the private allowlisted bundle, then run both cells. All 11 methods use 360 s / 1,000,000 expansions per puzzle. One session evaluates a five-map shard; 100 maps require multiple sessions. No training or new checkpoint conversion.


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, zipfile

# Upload the allowlisted bundle as a PRIVATE Kaggle dataset and attach it.
BUNDLE = None  # Optional explicit bundle zip OR auto-extracted input directory.
START_MAP, END_MAP = 1, 5  # Change to disjoint ranges for later sessions.
RESUME_DIR = None  # Optional attached directory containing manifest.json/results.jsonl.
PROJECT = Path('/kaggle/working/sokoban7_original')
OUT = Path(f'/kaggle/working/7box_maps{START_MAP:03d}_{END_MAP:03d}')
if BUNDLE is None:
    matches = list(Path('/kaggle/input').rglob('sokoban7_original_protocol.zip'))
    if not matches:
        matches = [p.parent for p in Path('/kaggle/input').rglob('BUNDLE_SHA256.json')]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one bundle; found {len(matches)}. Set BUNDLE explicitly.')
    BUNDLE = matches[0]
BUNDLE = Path(BUNDLE)
PROJECT.mkdir(parents=True, exist_ok=True)
def install_bundle(read, hashes):
    for name, expected in hashes.items():
        dest = (PROJECT / name).resolve()
        if not dest.is_relative_to(PROJECT.resolve()):
            raise RuntimeError('Unsafe bundle path')
        data = read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise RuntimeError(f'Bundle checksum mismatch: {name}')
        if dest.exists() and dest.read_bytes() != data:
            raise RuntimeError(f'Existing source differs: {name}')
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
if BUNDLE.is_dir():
    hashes = json.loads((BUNDLE / 'BUNDLE_SHA256.json').read_text())
    def read_input(name):
        source = (BUNDLE / name).resolve()
        if not source.is_relative_to(BUNDLE.resolve()):
            raise RuntimeError('Unsafe input path')
        return source.read_bytes()
    install_bundle(read_input, hashes)
else:
    with zipfile.ZipFile(BUNDLE) as archive:
        hashes = json.loads(archive.read('BUNDLE_SHA256.json'))
        if set(archive.namelist()) != set(hashes) | {'BUNDLE_SHA256.json'}:
            raise RuntimeError('Unexpected bundle files')
        install_bundle(archive.read, hashes)
if RESUME_DIR is not None:
    if OUT.exists():
        raise RuntimeError('Resume output already exists; inspect before copying')
    shutil.copytree(RESUME_DIR, OUT)
    # A copied input snapshot's lock belongs to the previous job, not this one.
    (OUT / 'run/.run.lock').unlink(missing_ok=True)
    if (OUT / 'data').exists():
        for source in (OUT / 'data').glob('sokoban_7box_original_test.*'):
            dest = PROJECT / 'data' / source.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            if dest.exists() and dest.read_bytes() != source.read_bytes():
                raise RuntimeError('Resume dataset conflicts with bundled dataset')
            shutil.copy2(source, dest)
print('Verified original-protocol bundle. Output:', OUT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'gym-sokoban==0.0.6'], check=True)
command = [sys.executable, '-u', str(PROJECT / 'tools/run_sokoban7_cloud.py'),
           '--start-map', str(START_MAP),
           '--end-map', str(END_MAP), '--wall-hours', '7.5', '--output', str(OUT)]
try:
    subprocess.run(command, cwd=PROJECT, check=True)
finally:
    if OUT.exists():
        archive_path = shutil.make_archive(str(OUT), 'zip', OUT)
        print('Download this result archive:', archive_path)
